In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Sample data
data = {
    "ID": range(1, 11),
    "loan_candidate": [
        "Emily Chen",
        "Ryan Thompson",
        "Sophia Patel",
        "Oliver Lee",
        "Ava Morales",
        "Ethan Hall",
        "Isabella Garcia",
        "Lucas Brooks",
        "Mia Davis",
        "Noah Martin"
    ],
    "loan_surveyor_comment": [
        "Property value matches loan amount, approved",
        "Insufficient collateral provided, rejected",
        "Credit history checks out, pending verification",
        "Income documents incomplete, request resubmission",
        "Business plan looks solid, approved in principle",
        "Loan amount exceeds property value, rejected",
        "Client credit score is excellent, fast-tracked",
        "Incomplete employment history, request update",
        "Vehicle offered as collateral is acceptable",
        "Loan application pending manager's review"
    ]
}

# Create DataFrame
df = pd.DataFrame(data)

# Define phrases
phrases = [
    "approved",
    "rejected",
    "pending",
    "collateral",
    "credit history"
]

# Create new columns based on phrases
# Load a pre-trained sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode the phrases
phrase_embeddings = model.encode(phrases, convert_to_tensor=True)

# Function to check if any phrase is present in the comment
def check_phrases(comment):
    # Split comment into words and encode each word
    words = comment.split()
    word_embeddings = model.encode(words, convert_to_tensor=True)
    
    # Compute cosine similarities for each word against each phrase
    max_similarities = []
    for phrase_emb in phrase_embeddings:
        # Get similarities between each word and the current phrase
        similarities = util.pytorch_cos_sim(word_embeddings, phrase_emb.unsqueeze(0))
        # Take the maximum similarity score (best matching word)
        max_sim = similarities.max().item()
        max_similarities.append(1 if max_sim > 0.7 else 0)
    
    return max_similarities

# Apply the function to each comment
for i, phrase in enumerate(phrases):
    df[f"comment_{phrase.replace(' ', '_').lower()}"] = df["loan_surveyor_comment"].apply(lambda x: check_phrases(x)[i])

# Print updated DataFrame
print(df)

   ID   loan_candidate                              loan_surveyor_comment  \
0   1       Emily Chen       Property value matches loan amount, approved   
1   2    Ryan Thompson         Insufficient collateral provided, rejected   
2   3     Sophia Patel    Credit history checks out, pending verification   
3   4       Oliver Lee  Income documents incomplete, request resubmission   
4   5      Ava Morales   Business plan looks solid, approved in principle   
5   6       Ethan Hall       Loan amount exceeds property value, rejected   
6   7  Isabella Garcia     Client credit score is excellent, fast-tracked   
7   8     Lucas Brooks      Incomplete employment history, request update   
8   9        Mia Davis        Vehicle offered as collateral is acceptable   
9  10      Noah Martin          Loan application pending manager's review   

   comment_approved  comment_rejected  comment_pending  comment_collateral  \
0                 1                 0                0                   0

In [23]:
import pandas as pd

# Sample data
data = {
    "ID": range(1, 11),
    "loan_candidate": [
        "Emily Chen",
        "Ryan Thompson",
        "Sophia Patel",
        "Oliver Lee",
        "Ava Morales",
        "Ethan Hall",
        "Isabella Garcia",
        "Lucas Brooks",
        "Mia Davis",
        "Noah Martin"
    ],
    "loan_surveyor_comment": [
        "Property value matches loan amount, approved",
        "Insufficient collateral provided, rejected",
        "Credit history checks out, pending verification",
        "Income documents incomplete, request resubmission",
        "Business plan looks solid, approved in principle",
        "Loan amount exceeds property value, rejected",
        "Client credit score is excellent, fast-tracked",
        "Incomplete employment history, request update",
        "Vehicle offered as collateral is acceptable",
        "Loan application pending manager's review"
    ]
}

# Create DataFrame
df = pd.DataFrame(data)

# Define phrases
phrases = [
    "approved",
    "rejected",
    "pending",
    "collateral",
    "credit history"
]

# Create new columns
for phrase in phrases:
    df[f"comment_{phrase.replace(' ', '_').lower()}"] = df["loan_surveyor_comment"].str.contains(phrase, case=False).astype(int)


# Print updated DataFrame
print(df)

   ID   loan_candidate                              loan_surveyor_comment  \
0   1       Emily Chen       Property value matches loan amount, approved   
1   2    Ryan Thompson         Insufficient collateral provided, rejected   
2   3     Sophia Patel    Credit history checks out, pending verification   
3   4       Oliver Lee  Income documents incomplete, request resubmission   
4   5      Ava Morales   Business plan looks solid, approved in principle   
5   6       Ethan Hall       Loan amount exceeds property value, rejected   
6   7  Isabella Garcia     Client credit score is excellent, fast-tracked   
7   8     Lucas Brooks      Incomplete employment history, request update   
8   9        Mia Davis        Vehicle offered as collateral is acceptable   
9  10      Noah Martin          Loan application pending manager's review   

   comment_approved  comment_rejected  comment_pending  comment_collateral  \
0                 1                 0                0                   0

In [9]:
import pandas as pd
import dask.dataframe as dd

# Load the Excel file
file_path = 'question_answer_all_imove.xlsx'
df = pd.read_excel(file_path)

# Convert to Dask DataFrame
df_dask = dd.from_pandas(df, npartitions=4) 

In [ ]:
df_dask= df_dask.persist()  # Persist the Dask DataFrame in memory

print(df_dask.head())  # Display the first few rows of the Dask DataFrame

# Perform some operations on the Dask DataFrame
# For example, calculate the mean of a column
max_value_order_id = df_dask['ORDER_ID'].max().compute()  # Compute the max of the 'ORDER_ID' column
print(f"Max ORDER_ID: {max_value_order_id}")

   ORDER_ID               QUESTION ANSWERTEXT ANSWER_VALUE
0         0  Nama Sesuai Identitas          0         <NA>
1         0       Tempat Kelahiran          0         <NA>
2         0         No Hp Konsumen          0         <NA>
3         0             Company Id       <NA>          FIF
4         0          Kondisi Rumah       <NA>      TERAWAT
Max ORDER_ID: 12825001611
